# 03 — Backtesting Walk-Forward + Simulación de Merma (OE4)

Validación del sistema predictivo según el **Objetivo Específico 4 (OE4)** de la tesis:

1. **Backtesting walk-forward deslizante** sobre la historia sintética (escenario Óptimo, ~730 días).
2. **Simulación de merma (spoilage)** respetando la vida útil por producto.
3. **Comparativa contra la política baseline del pastelero** (media móvil + buffer de sobreproducción).
4. Reporte de **reducción de merma %** y **mejora del nivel de servicio (fill rate)**.

> **Limitación declarada (§3.8 de la tesis):** los datos provienen de `seed.py`, cuyo generador ya codifica las reglas que el modelo captura. Esto puede inflar las métricas de precisión. La comparativa baseline-vs-sistema, sin embargo, **sí** es informativa porque ambas políticas enfrentan la misma demanda real.

Este notebook consume la BD sintética ya sembrada en `src/casserisissima.db`.

In [ ]:
import os, sys
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
sys.path.insert(0, SRC_DIR)

# Fijar la BD sembrada en src/ (database.py usa ruta relativa + load_dotenv en CWD)
os.environ.setdefault('DATABASE_URL', 'sqlite:///' + os.path.join(SRC_DIR, 'casserisissima.db'))

import pandas as pd
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 200)

from core.ml.backtest_report import run_full_report, write_csv_outputs
print('OK — módulos OE4 importados.')

## 1) Ejecutar el reporte

Para mantener el notebook ágil se evalúan los **3 primeros productos** del escenario Óptimo.
El reporte completo (12 productos) se regenera con:
`python scripts/regenerar_backtest.py`

In [ ]:
reporte = run_full_report(
    scenario_id=2,           # Óptimo (May 2024 – May 2026, ~730 días)
    max_products=3,          # subconjunto ágil; quitar para run completo
    train_window_days=180,   # ventana deslizante de 180 días
    horizon=14,              # horizonte de 2 semanas por paso
    retrain_every=7,         # reentrenar 1x por semana
    baseline_k=7,            # baseline: media móvil de 7 días
    baseline_buffer=0.10,    # baseline: +10% sobreproducción
)
print(reporte['summary'])

## 2) Tabla agregada por producto

Métricas de pronóstico + merma + fill rate, por producto y por política.

In [ ]:
agg_df = pd.DataFrame(reporte['aggregated_table'])
agg_df[['name','sku','shelf_life_days','n_windows','n_predictions',
        'mae','mape','rmse',
        'waste_pct_system','waste_pct_baseline','waste_reduction_pct',
        'fill_rate_system','fill_rate_baseline','fill_rate_delta']]

## 3) Detalle por producto (comparativa baseline vs sistema)

In [ ]:
filas = []
for pid, info in reporte['per_product'].items():
    filas.append({
        'producto': info['name'],
        'sku': info['sku'],
        'vida_util_dias': info['shelf_life_days'],
        'ventanas': info['n_windows'],
        'predicciones': info['n_predictions'],
        **info['comparison'],
    })
pd.DataFrame(filas)

## 4) Agregados medios sobre el subconjunto evaluado

In [ ]:
pd.DataFrame([reporte['aggregated']])

## 5) Persistir resultados en `results/`

CSVs + JSON de resumen, consumibles por el capítulo de resultados de la tesis.

In [ ]:
paths = write_csv_outputs(reporte)
for p in paths:
    print('escrito:', p)